In [ ]:
# @colab-setup
# Run this cell first. On Colab it installs the deps; locally it is a no-op.
import sys
if "google.colab" in sys.modules:
    %pip install -q pyobis pandas folium


# 01 — pyobis Quickstart

**Time budget:** ~10 min · **Goal:** show the lowest-friction Python path to OBIS data, and be honest about its limits.

[`pyobis`](https://github.com/iobis/pyobis) is a thin Python client over the OBIS REST API. Three modules cover most needs:

- `pyobis.occurrences` — occurrence records (a single observation in space and time)
- `pyobis.dataset` — dataset metadata (titles, contacts, extents, …)
- `pyobis.checklist` — taxonomic coverage

Demo dataset for the rest of the series:

```
DEMO_UUID = 'd895e645-a98d-4720-b6fb-332929190f36'   # Maritimes Spring RV Surveys
```

In [1]:
# If running outside the project's uv env:
# %pip install pyobis pandas folium

from pyobis import occurrences, dataset, checklist
import pandas as pd

DEMO_UUID = 'd895e645-a98d-4720-b6fb-332929190f36'  # Maritimes Spring RV Surveys

## 1. Search occurrences

Pull 200 *Calanus finmarchicus* records from a rough Canadian-waters bounding polygon.

Heads-up: OBIS's `country=` parameter does **not** filter geographically — it filters by OBIS *node*, which is rarely what you want. Use `geometry=` (a WKT polygon) for spatial filtering.

In [4]:
CANADA_BBOX_WKT = 'POLYGON((-141 41, -52 41, -52 84, -141 84, -141 41))'

query = occurrences.search(
    scientificname='Calanus finmarchicus',
    geometry=CANADA_BBOX_WKT,
    size=200,
)
df = pd.DataFrame(query.execute())
print(f'{len(df)} records')
df[['scientificName', 'decimalLatitude', 'decimalLongitude', 'eventDate', 'datasetID']].head()

2026-04-28 09:09:50 - pyobis.cache.cache - INFO - Cache initialized at /home/richard/.cache/pyobis
2026-04-28 09:09:50 - pyobis.obisutils - INFO - 200 to be fetched. Estimated time =0.0039488720893859860 seconds
2026-04-28 09:09:50 - pyobis.obisutils - INFO - Fetching: [████████████████████████████████████████████████████████████████████████████████████████████████████] 200/200
2026-04-28 09:09:50 - pyobis.cache.cache - INFO - Cache initialized at /home/richard/.cache/pyobis
2026-04-28 09:09:50 - pyobis.obisutils - INFO - Fetched 200 records.


200 records


,scientificName,decimalLatitude,decimalLongitude,eventDate,datasetID
0,Calanus finmarchicus,43.690000,-64.450000,NaN,NaN
1,Calanus finmarchicus,46.503000,-52.853000,2017-01-17 23:29:00+00,NaN
2,Calanus finmarchicus,45.320000,-58.590000,2003-03-09T12:00:00Z,NaN
3,Calanus finmarchicus,43.870000,-62.910000,NaN,DFO_BioChem_Sameoto
4,Calanus finmarchicus,43.779999,-57.849998,2003-04-16T12:00:00Z,NaN


## 2. Quick map

Plot the points on a slippy map with `folium` to confirm we got the geography we asked for.

In [5]:
import folium

geo = df.dropna(subset=['decimalLatitude', 'decimalLongitude'])
centre = [geo['decimalLatitude'].mean(), geo['decimalLongitude'].mean()]
m = folium.Map(location=centre, zoom_start=4, tiles='cartodbpositron')
for _, r in geo.iterrows():
    folium.CircleMarker(
        location=[r['decimalLatitude'], r['decimalLongitude']],
        radius=2,
        weight=0,
        fill=True,
        fill_opacity=0.6,
        popup=r.get('eventDate'),
    ).add_to(m)
m

## 3. Find datasets carrying a species

`dataset.search` accepts the same filters as `occurrences.search` and gives you the *datasets* that contain matching records — useful when you want to follow the data back to its source.

In [6]:
ds = pd.DataFrame(
    dataset.search(scientificname='Calanus finmarchicus', size=10).execute()
)
cols = [c for c in ('id', 'title', 'records') if c in ds.columns]
ds[cols].head(10)

2026-04-28 09:09:58 - pyobis.cache.cache - INFO - Cache initialized at /home/richard/.cache/pyobis


""
0
1
2
3
4
5
6
7
8
9


## 4. Single-dataset metadata

Pull the full record for our demo UUID.

In [7]:
meta = dataset.get(id=DEMO_UUID).execute()
if isinstance(meta, dict) and 'results' in meta:
    meta = meta['results'][0]
{k: meta.get(k) for k in ('title', 'abstract', 'records', 'extent') if k in meta}

2026-04-28 09:10:12 - pyobis.cache.cache - INFO - Cache initialized at /home/richard/.cache/pyobis


{'title': 'Maritimes Spring Research Vessel Surveys',
 'abstract': 'The Fisheries and Oceans Canada (DFO) ecosystem surveys consist of research vessel survey data collected to monitor the distribution and abundance of fish and invertebrates throughout the Scotian Shelf, Bay of Fundy and Georges Bank. The surveys follow a stratified random sampling design, and include sampling of fish and invertebrates using a bottom otter trawl. These survey data are the primary data source for monitoring trends in species distribution, abundance, and biological condition within the region, and also provide data to the Atlantic Zonal Monitoring Program (AZMP) for monitoring hydrographic variability. Collected data includes total catch in numbers and weights by species. Length frequency data is available for most species, as are the age, sex, maturity and weight information for a subset of the individual animals. Other data such as ageing material, genetic material, and stomach contents are often also c

## 5. Taxonomic checklist

Which taxa are in OBIS under the genus *Calanus*?

In [8]:
tax = pd.DataFrame(checklist.list(scientificname='Calanus', size=20).execute())
cols = [c for c in ('scientificName', 'taxonRank', 'records') if c in tax.columns]
tax[cols].head(20)

2026-04-28 09:10:16 - pyobis.cache.cache - INFO - Cache initialized at /home/richard/.cache/pyobis
2026-04-28 09:10:16 - pyobis.obisutils - INFO - Estimated records: 18
2026-04-28 09:10:16 - pyobis.obisutils - INFO - Fetching: [████████████████████████████████████████████████████████████████████████████████████████████████████] 18
2026-04-28 09:10:16 - pyobis.cache.cache - INFO - Cache initialized at /home/richard/.cache/pyobis


TypeError: Logger._log() got an unexpected keyword argument 'end'

## When pyobis stops being the right tool

- pyobis hides pagination — it's quietly making `&size=…&after=…` requests under the hood.
- For datasets with **>10 000 records**, this gets slow (minutes).
- For analytical work — group-bys, joins, spatial filters — pulling the whole dataset row-by-row is wasteful.

**Switch to:**

- **Notebook 2** — raw REST when you need fine-grained control over pagination, fields, faceting.
- **Notebook 3** — Parquet on S3 when you want speed (typically 5–20× faster) and SQL.
- **Notebook 4** — `cioos_metadata_conversion` when you want a CIOOS-shaped metadata record, not occurrences.